# Notebook 02 — Train one SD2.1 facial-acne LoRA on Kaggle (T4)
This notebook prepares the **train-only** portion of your real acne images and captions, performs provisional duplicate grouping, and invokes the official pinned Hugging Face Diffusers v0.35.1 LoRA trainer. **No subtype labels are invented.**

**Scope:** General facial acne adaptation. `TRAIN_WITH_CYSTIC_TI=True` is an optional *experimental* branch only for individually annotated cystic examples; by default train **without** it, then combine with the published TI at inference. Without GPU/network Kaggle, CUDA training remains unverified.


## 1. Kaggle setup
Select T4 GPU + Internet ON. Add your 1,000-image Kaggle dataset from **Add Input**. Set `IMAGE_ROOT` below to your dataset folder. If a `train/` folder exists, this notebook only uses it. Otherwise supply a global `SPLIT_CSV` or the notebook will create a 70/15/15 acne-only group split to protect held-out samples.

If your dataset has captions, optionally provide a CSV with columns `filename,caption`, where `filename` matches the image basename or path relative to `IMAGE_ROOT`. No medical characteristics are guessed from pixels.

In [ ]:
# Dependencies are pinned to match the official Diffusers v0.35.1 LoRA example.
# Run in a fresh Kaggle session. Do NOT replace Kaggle's CUDA-enabled PyTorch.
import sys, subprocess, importlib.metadata as metadata
PACKAGES = [
    "diffusers==0.35.1", "transformers==4.56.2", "accelerate==1.10.1",
    "peft==0.17.1", "datasets==3.6.0", "huggingface_hub==0.34.4",
    "safetensors==0.6.2", "ftfy==6.3.1", "imagehash==4.3.2",
    "tensorboard==2.20.0",  # Required by --report_to tensorboard in Notebook 02
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
for spec in PACKAGES:
    package, expected = spec.split('==')
    got = metadata.version(package)
    if got != expected:
        raise RuntimeError(f'Package mismatch: {package} {got}; expected {expected}. Restart Kaggle session then rerun.')
print('Pinned package versions verified. If you previously imported a conflicting library, restart the session now.')


In [ ]:
from pathlib import Path
import sys, os, csv, json, hashlib, shutil, subprocess, random, math
import numpy as np
from PIL import Image, ImageOps
import torch, diffusers, datasets, transformers, accelerate, peft
from IPython.display import display
assert torch.cuda.is_available(), 'Set Kaggle accelerator to T4 or P100 before training.'
assert diffusers.__version__ == '0.35.1', 'Restart kernel after pip install; wrong Diffusers imported.'
assert transformers.__version__ == '4.56.2', 'Restart kernel after pip install; wrong Transformers imported.'
# Official training script imports torchvision. Verify early rather than failing after checkpoint downloads.
try:
    import torchvision
    from torchvision import transforms
except Exception as exc:
    raise RuntimeError('Torchvision import failed. Your Kaggle torch/torchvision build may be mismatched; use a fresh GPU session.') from exc
print('GPU',torch.cuda.get_device_name(0))
print('torch',torch.__version__,'torchvision',torchvision.__version__,'diffusers',diffusers.__version__)


## 2. Kaggle v2 real-data source and approved train split
Only `/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection` is allowed as the original real image collection. The cell below explicitly points to it, but **the actual LoRA training images are staged only from the human-reviewed frozen real TRAIN manifest**. Attach `real_all_manifest.csv`, the saved deduplicated split images, and the passed label-audit JSON. The notebook refuses raw unsplit images, missing approvals and test/validation leakage. Training logs and weights remain in the Kaggle Output folder; do not commit private images or large checkpoints to GitHub automatically.


**Updated split protocol:** this notebook explicitly reports the selected class's train/validation/test counts from the already reviewed GLOBAL image-group split, saves `class_frozen_split_counts.csv` and the private assignment CSV, and trains LoRA **only** on that class's training images. It will not create an independent raw-folder split that could leak repeated patients across classes. Complete console output is in `notebook_console.log` and the actual trainer log.


In [ ]:
IMAGE_ROOT = Path('/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection/Face_Skin_dataset/Acne')  # CHANGE THIS
SPLIT_CSV = ''        # Optional CSV: filename,split (globally frozen splits preferred)
CAPTIONS_CSV = ''     # Optional CSV: filename,caption; else general honest acne caption
AUTO_SPLIT = False    # Provisional group split when no train/ directory or split CSV exists
BASE_MODEL = 'sd2-community/stable-diffusion-2-1-base' # Unofficial compatible SD2.1 512-base mirror
TI_MODEL = 'hcarrion/acne-cystic'
TRAIN_WITH_CYSTIC_TI = False  # Only if captioned cystic-acne images are verified
OUT = Path('/kaggle/working/acne_lora_output')
DATA_PREP = Path('/kaggle/working/acne_lora_training_data')
STEPS = 20    # safe first GPU smoke test; raise only after review
RANK = 16
RESOLUTION = 512
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 1e-4
SEED = 34021
CHECKPOINT_STEPS = 250
USE_8BIT_ADAM = False  # Standard AdamW avoids bitsandbytes dependency; LoRA fits T4 more easily
assert RESOLUTION % 8 == 0 and STEPS > 0 and RANK > 0
if not IMAGE_ROOT.exists():
    raise FileNotFoundError(f'Change IMAGE_ROOT to your added Kaggle dataset path. Current: {IMAGE_ROOT}')
OUT.mkdir(parents=True,exist_ok=True)
DATA_PREP.mkdir(parents=True,exist_ok=True)
print('Using dataset folder:',IMAGE_ROOT)


In [ ]:
# SOURCE: sole Kaggle Facial Skin v2 collection; actual LoRA input is the APPROVED real train split ONLY.
from pathlib import Path
import sys,subprocess
DERMSYNREAL_REPO=Path('/kaggle/working/DermSynReal')
if not (DERMSYNREAL_REPO/'src/dermsynreal/kaggle_runs.py').is_file():
    subprocess.check_call(['git','clone','--depth','1','https://github.com/shanmukhdatta/DermSynReal.git',str(DERMSYNREAL_REPO)])
sys.path.insert(0,str(DERMSYNREAL_REPO/'src'))
from dermsynreal.kaggle_runs import prepare_lora_train_only,append_run_log
RAW_REAL_DATASET_V2='/kaggle/input/datasets/shanmukhadattaboda/facial-skin-conditions-real-image-collection'
REAL_TRAIN_MANIFEST='/kaggle/input/EDIT_APPROVED_REAL_SPLIT/real_all_manifest.csv'
REAL_TRAIN_IMAGE_ROOT='/kaggle/input/EDIT_APPROVED_REAL_IMAGES/facial_real_split_v2'
PASSED_REAL_LABEL_AUDIT='/kaggle/input/EDIT_APPROVED_REAL_AUDIT/label_audit_summary.json'
IMAGE_ROOT,real_train_evidence=prepare_lora_train_only(
    real_split_manifest=REAL_TRAIN_MANIFEST,real_image_root=REAL_TRAIN_IMAGE_ROOT,
    cls='Acne',stage_dir='/kaggle/working/approved_Acne_lora_input',
    origin_root=RAW_REAL_DATASET_V2,label_audit_json=PASSED_REAL_LABEL_AUDIT)
AUTO_SPLIT=False
ALLOW_PROVISIONAL_SPLIT=False
SPLIT_CSV=''
# The global group-safe split is reused; show this class's training/validation/test counts.
print('Class split (before subtype selection):',real_train_evidence['class_frozen_split_counts'])
from dermsynreal.kaggle_runs import start_notebook_logging
PRIVATE_LORA_CONSOLE=start_notebook_logging(OUT,kind='lora_Acne',
    config={'class_split_counts':real_train_evidence['class_frozen_split_counts'],
            'train_only_image_count':real_train_evidence['count'],
            'split_sha256':real_train_evidence['manifest_sha256']},repo_root=DERMSYNREAL_REPO)
print('Research console log:',PRIVATE_LORA_CONSOLE)
print('Approved real training images:',real_train_evidence['count'],'Folder:',IMAGE_ROOT)
# Capture full trainer stdout/stderr in an append-only research log while streaming to Kaggle.
def run_logged(command,path):
    import subprocess
    from pathlib import Path
    log=Path(path);log.parent.mkdir(parents=True,exist_ok=True)
    with log.open('a',encoding='utf-8',errors='replace') as f:
        proc=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,errors='replace',bufsize=1)
        for line in proc.stdout:
            print(line,end='');f.write(line);f.flush()
        code=proc.wait()
    if code:raise subprocess.CalledProcessError(code,command)


## 3. Read and validate acne images, compute duplicate groups
This scans JPG/PNG/WebP, checks images can be decoded, and computes a canonical flip-aware pHash. For unsplit data, it clusters likely near duplicates (Hamming distance ≤5) before assigning provisional train/validation/test sets. **Review this split**, because near-duplicate detection alone does not establish patient independence. The output lists excluded unreadable files.

In [ ]:
import imagehash
from sklearn.model_selection import GroupShuffleSplit
EXTS={'.jpg','.jpeg','.png','.webp','.bmp'}
root_train = IMAGE_ROOT/'train'
using_train_dir = root_train.is_dir()
scan_root = root_train if using_train_dir else IMAGE_ROOT
paths = sorted(p for p in scan_root.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
if not paths: raise RuntimeError('No images found. Verify IMAGE_ROOT and whether images are nested in subfolders.')

entries=[]; invalid=[]
for p in paths:
    try:
        with Image.open(p) as img:
            img=ImageOps.exif_transpose(img).convert('RGB')
            if img.width<128 or img.height<128: raise ValueError('image too small')
            hvals = [int(str(imagehash.phash(t)),16) for t in (
                img,ImageOps.mirror(img),ImageOps.flip(img),ImageOps.mirror(ImageOps.flip(img))
            )]
            digest=hashlib.sha256(img.tobytes()).hexdigest()
        entries.append({'path':p,'sha256':digest,'phash':min(hvals)})
    except Exception as exc:
        invalid.append((str(p),str(exc)))
if not entries: raise RuntimeError('All discovered images failed decoding or minimum-size checks')
print('Images',len(entries),'invalid',len(invalid))
for p,err in invalid[:10]: print('BAD',p,err)

# Union-find combines exact repeats and likely flip/crop/recompress near-copies.
n=len(entries); parent=list(range(n))
def find(i):
    while parent[i]!=i:
        parent[i]=parent[parent[i]]; i=parent[i]
    return i
def unite(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb: parent[rb]=ra
for a in range(n):
    for b in range(a+1,n):
        if entries[a]['sha256']==entries[b]['sha256'] or (entries[a]['phash'] ^ entries[b]['phash']).bit_count() <= 5:
            unite(a,b)
for i,e in enumerate(entries): e['group']=find(i)
print('Near-duplicate groups:',len({e['group'] for e in entries}))

## 4. Use train-only partition and create captions
If a supplied split CSV exists it overrides automatic splitting and `train/` selection. Otherwise `train/` counts as explicitly train-only; otherwise a reproducible 70/15/15 pHash-group split is created. Save `acne_data_manifest.csv` and **freeze it** for future runs.

When captions are missing, the notebook uses a general, medically non-specific acne caption; detailed morphological supervision requires manual labels.

In [ ]:
def read_lookup(csv_path,value_key):
    if not csv_path: return {}
    with open(csv_path,encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    if not rows or not {'filename',value_key}.issubset(rows[0]):
        raise ValueError(f'{csv_path}: columns filename,{value_key} required')
    return {r['filename'].replace('\\','/'):r[value_key] for r in rows}

split_map=read_lookup(SPLIT_CSV,'split')
caption_map=read_lookup(CAPTIONS_CSV,'caption')
if split_map:
    for e in entries:
        rel=e['path'].relative_to(IMAGE_ROOT).as_posix()
        e['split']=split_map.get(rel,split_map.get(e['path'].name,'exclude')).strip().lower()
    if any(e['split'] not in {'train','val','valid','validation','test','exclude'} for e in entries):
        raise ValueError('Bad split values in SPLIT_CSV')
    print('Using supplied split CSV')
elif using_train_dir:
    for e in entries: e['split']='train'
    print('Using existing train/ partition only. No validation/test files are read.')
else:
    if not AUTO_SPLIT: raise RuntimeError('Supply a train/ directory or SPLIT_CSV; AUTO_SPLIT is False.')
    idx=np.arange(len(entries)); groups=[e['group'] for e in entries]
    if len(set(groups))<4: raise RuntimeError('Too few independent groups to split; provide a reviewed train/ partition.')
    first=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
    pool, test = next(first.split(idx,groups=groups))
    second=GroupShuffleSplit(n_splits=1,test_size=0.15/0.85,random_state=SEED+1)
    train_rel,val_rel=next(second.split(pool,groups=[groups[i] for i in pool]))
    part={int(i):'train' for i in pool[train_rel]}
    part.update({int(i):'val' for i in pool[val_rel]})
    part.update({int(i):'test' for i in test})
    for i,e in enumerate(entries):e['split']=part[i]
    print('Created provisional group-aware split. Do NOT combine with a separate global test set without checking overlap.')

# Every duplicate group must belong to one partition, even for supplied manifests.
group_assign={}
for e in entries:
    spl=e['split']; g=e['group']
    if g in group_assign and group_assign[g]!=spl:
        raise RuntimeError('Duplicate group spans splits; revise split manifest before training')
    group_assign[g]=spl

for e in entries:
    rel=e['path'].relative_to(IMAGE_ROOT).as_posix()
    caption=caption_map.get(rel,caption_map.get(e['path'].name,'')).strip()
    e['caption']=caption or 'clinical photograph of a face with acne, natural facial skin texture'
    if '<acne-cystic-class>' in e['caption'] and not TRAIN_WITH_CYSTIC_TI:
        raise ValueError('TI token found in caption but TRAIN_WITH_CYSTIC_TI=False')
train_entries=[e for e in entries if e['split']=='train']
# Keep one representative per near-duplicate group for LoRA; manifest retains full provenance.
# NOTE: pHash groups are provisional and should be manually audited for clinical photos.
train_count_before_dedup=len(train_entries)
seen_train_groups=set()
train_entries_unique=[]
for e in train_entries:
    if e['group'] not in seen_train_groups:
        train_entries_unique.append(e)
        seen_train_groups.add(e['group'])
train_entries=train_entries_unique
print('Removed',train_count_before_dedup-len(train_entries),'repeat/near-duplicate training images')
if not train_entries: raise RuntimeError('No training images selected; inspect split CSV and paths')
if TRAIN_WITH_CYSTIC_TI:
    if not CAPTIONS_CSV:
        raise RuntimeError('Cystic TI training requires a manually reviewed CAPTIONS_CSV; general acne labels are insufficient.')
    count=sum('<acne-cystic-class>' in e['caption'] for e in train_entries)
    if count==0: raise RuntimeError('No confirmed cystic examples with <acne-cystic-class> in captions')
    print('Using published cystic-acne TI for',count,'explicitly labeled training examples only.')

manifest_path=OUT/'acne_data_manifest.csv'
with manifest_path.open('w',newline='',encoding='utf-8') as f:
    writer=csv.DictWriter(f,fieldnames=['filename','split','group_id','sha256','canonical_phash','caption'])
    writer.writeheader()
    for e in entries:
        writer.writerow({'filename':e['path'].relative_to(IMAGE_ROOT).as_posix(),
                         'split':e['split'],'group_id':e['group'],'sha256':e['sha256'],
                         'canonical_phash':format(e['phash'],'016x'),'caption':e['caption']})
from collections import Counter
print('Partition counts:',dict(Counter(e['split'] for e in entries)))
print('Train count:',len(train_entries),'Manifest:',manifest_path)


## 5. Build imagefolder training data
This creates a `metadata.jsonl` file in the format required by the official Diffusers trainer. Images are padded to square rather than cropped, keeping the full face visible. The neutral padding is a consistent background but may itself become a generator artifact—inspect the sample preview. Copies are stored only under `/kaggle/working/`.

In [ ]:
import shutil
# Clear stale imagefolder files so relaunching a session cannot blend datasets.
for old in DATA_PREP.glob('image_*.*'): old.unlink()
meta_file=DATA_PREP/'metadata.jsonl'
with meta_file.open('w',encoding='utf-8') as dst:
    for idx,e in enumerate(train_entries):
        with Image.open(e['path']) as original:
            im=ImageOps.exif_transpose(original).convert('RGB')
            im=ImageOps.pad(im,(RESOLUTION,RESOLUTION),method=Image.Resampling.LANCZOS,color=(128,128,128))
        name=f'image_{idx:06d}.jpg'
        im.save(DATA_PREP/name,'JPEG',quality=95)
        dst.write(json.dumps({'file_name':name,'text':e['caption']},ensure_ascii=False)+'\n')
print('Prepared',len(train_entries),'training images and',meta_file)
with Image.open(DATA_PREP/'image_000000.jpg') as img: display(img.resize((256,256)))

## 6. Fetch the *fixed release* of the official Diffusers training script
The notebook downloads the `v0.35.1` script from the official GitHub tag, checks its expected API markers, and (only if requested) inserts a small **explicit textual-inversion loader** before its frozen text encoder is created. This retains the official optimizer, noise schedule, loss, saving and LoRA conventions. The experimental TI insertion supports a **single-vector** embedding as used by this repository; other formats fail visibly instead of silently training the wrong concept.

The public TI embedding is a *conditioning token*, not an initial LoRA matrix. For your unlabeled 1,000-image set, keep `TRAIN_WITH_CYSTIC_TI=False` and combine TI with the trained LoRA later at inference.

In [ ]:
from urllib.request import urlopen
SCRIPT_URL='https://raw.githubusercontent.com/huggingface/diffusers/v0.35.1/examples/text_to_image/train_text_to_image_lora.py'
with urlopen(SCRIPT_URL,timeout=90) as r:
    script=r.read().decode('utf-8')
assert 'check_min_version("0.35.0")' in script, 'Official script unexpectedly changed'
assert 'StableDiffusionPipeline.save_lora_weights(' in script, 'Script must save Diffusers-format LoRA'
assert '    # freeze parameters of models to save more memory' in script, 'Trainer structure unexpectedly changed'

if TRAIN_WITH_CYSTIC_TI:
    marker='    # freeze parameters of models to save more memory'
    if script.count(marker)!=1: raise RuntimeError('Official trainer changed; do not patch blindly.')
    # Insert after model/tokenizer/text-encoder are loaded and before encoder freezing.
    # Only explicitly labelled cystic images should mention the TI token in captions.
    patch='''    # BEGIN research extension: load cgDDI textual inversion into frozen CLIP encoder
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file as load_ti_safetensors
    ti_file = hf_hub_download("hcarrion/acne-cystic", filename="learned_embeds.safetensors")
    ti_state = load_ti_safetensors(ti_file)
    ti_token = "<acne-cystic-class>"
    if ti_token not in ti_state:
        raise RuntimeError(f"Expected {ti_token} in published textual-inversion file; keys: {list(ti_state)}")
    ti_embedding = ti_state[ti_token]
    if ti_embedding.ndim == 2 and ti_embedding.shape[0] == 1:
        ti_embedding = ti_embedding[0]
    if ti_embedding.ndim != 1 or ti_embedding.shape[0] != text_encoder.get_input_embeddings().weight.shape[1]:
        raise RuntimeError(f"Unexpected embedding shape: {ti_embedding.shape}")
    if tokenizer.add_tokens(ti_token) != 1:
        raise RuntimeError("Textual inversion token already present in the base tokenizer")
    text_encoder.resize_token_embeddings(len(tokenizer))
    with torch.no_grad():
        text_encoder.get_input_embeddings().weight[tokenizer.convert_tokens_to_ids(ti_token)].copy_(
            ti_embedding.to(dtype=text_encoder.get_input_embeddings().weight.dtype)
        )
    # END research extension
'''
    script=script.replace(marker,patch+marker)

trainer_path=Path('/kaggle/working/train_text_to_image_lora_sd21.py')
trainer_path.write_text(script,encoding='utf-8')
import ast
ast.parse(script)
print('Saved',trainer_path,'sha256:',hashlib.sha256(script.encode()).hexdigest())


## 7. Train LoRA using the official pinned script
**Default is standard PyTorch AdamW**, because the earlier file enabled 8-bit Adam without installing `bitsandbytes`. The pinned `tensorboard` package is included to satisfy `--report_to tensorboard`. If `USE_8BIT_ADAM=True` the launcher explicitly checks/installs its dependency. Use the same base model for inference.


In [ ]:
from huggingface_hub import model_info
import shlex, subprocess, time
base_commit = model_info(BASE_MODEL).sha
print("Training base revision",base_commit)
command=[
    sys.executable,'-m','accelerate.commands.launch',
    '--num_processes=1','--mixed_precision=fp16',
    str(trainer_path),
    '--pretrained_model_name_or_path',BASE_MODEL,
    '--revision',base_commit,
    '--mixed_precision','fp16',  # The official trainer checks args.mixed_precision to keep trainable LoRA weights FP32
    '--train_data_dir',str(DATA_PREP),
    '--image_column','image',
    '--caption_column','text',
    '--output_dir',str(OUT),
    '--resolution',str(RESOLUTION),
    '--train_batch_size',str(BATCH_SIZE),
    '--gradient_accumulation_steps',str(GRADIENT_ACCUMULATION),
    '--gradient_checkpointing',
    '--max_train_steps',str(STEPS),
    '--learning_rate',str(LEARNING_RATE),
    '--lr_scheduler','constant',
    '--lr_warmup_steps','0',
    '--rank',str(RANK),
    '--seed',str(SEED),
    '--checkpointing_steps',str(CHECKPOINT_STEPS),
    '--checkpoints_total_limit','3',
    '--dataloader_num_workers','0',
    '--report_to','tensorboard',
    '--center_crop',  # already square-padded, avoids random face cropping
]
if USE_8BIT_ADAM:
    try:
        import bitsandbytes
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'bitsandbytes==0.45.5'])
    command.append('--use_8bit_adam')
print('Training command:\n',shlex.join(command))
started=time.time()
run_logged(command,OUT/'trainer_console.log')
print('Training minutes:',round((time.time()-started)/60,1))


## 8. Verify final weights and export
Download this `.safetensors` file using Kaggle Output (or attach it as a Kaggle dataset to Notebook 01). Keep the base-model identifier and training metadata alongside it. Do not publish identifiable real-image examples without suitable permission.

In [ ]:
from safetensors import safe_open
weights=OUT/'pytorch_lora_weights.safetensors'
if not weights.exists():
    raise FileNotFoundError(f'Trainer did not produce {weights}. Check logs above.')
with safe_open(str(weights),framework='pt',device='cpu') as reader:
    keys=list(reader.keys())
print('LoRA file:',weights,'MB:',round(weights.stat().st_size/1024**2,2))
print('Tensor count:',len(keys),'Sample keys:',keys[:4])
assert keys and any('lora' in key.lower() for key in keys), 'Unexpected LoRA state dict.'
run_config={'base_model':BASE_MODEL,'textual_inversion':TI_MODEL if TRAIN_WITH_CYSTIC_TI else None,
            'train_with_cystic_ti':TRAIN_WITH_CYSTIC_TI,'resolution':RESOLUTION,'steps':STEPS,
            'rank':RANK,'learning_rate':LEARNING_RATE,'seed':SEED,
            'image_count':len(train_entries),'official_diffusers_script':'v0.35.1',
            'source_script_sha256':hashlib.sha256(script.encode()).hexdigest()}
(OUT/'training_config.json').write_text(json.dumps(run_config,indent=2))
import zipfile
zip_path=Path('/kaggle/working/acne_lora_export.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in [weights,OUT/'training_config.json',OUT/'acne_data_manifest.csv']:
        zf.write(p,arcname=p.name)
print('Export:',zip_path)
print('Next: set LORA_SOURCE to the extracted directory or safetensors path in Notebook 01.')

In [ ]:
# Retain the **three class split counts** beside the LoRA weight/checkpoint.
import shutil
from pathlib import Path
for _name in ('class_frozen_split_counts.csv','class_frozen_split_assignments.csv','lora_train_source_summary.json'):
    shutil.copy2(Path(IMAGE_ROOT)/_name,Path(OUT)/_name)
# Save durable metadata, model provenance and checksum manifest for the Kaggle Output tab.
from dermsynreal.kaggle_runs import append_run_log
append_run_log(OUT,kind='lora_fine_tuning',config={'class':'Acne',
    'base_model':BASE_MODEL,'training_steps':STEPS,'seed':SEED,
    'approved_real_train_count':real_train_evidence['count'],
    'approved_manifest_sha256':real_train_evidence['manifest_sha256']},
    artifacts=[OUT/'trainer_console.log',OUT# Notebook console continues being written; hash only after kernel stops.
        
        OUT/'class_frozen_split_counts.csv',OUT/'pytorch_lora_weights.safetensors'],
    repo_root=DERMSYNREAL_REPO)
print('LoRA output, logs and provenance:',OUT)

## References and limitations
- [Hugging Face Diffusers v0.35.1 official LoRA training script](https://github.com/huggingface/diffusers/blob/v0.35.1/examples/text_to_image/train_text_to_image_lora.py) (Apache-2.0)
- [Official Diffusers LoRA training guide](https://huggingface.co/docs/diffusers/training/lora)
- [cgDDI acne-cystic embedding](https://huggingface.co/hcarrion/acne-cystic)
- **Never use a real held-out image to train LoRA.** If this notebook created a provisional split, reconcile it with your global real-image grouping procedure before scientific results. No guarantee of subtype fidelity without subtype supervision. Run a real-data memorization audit after generation.